# Guided Lab: Database-Connected Agent

This lab builds a practical agent for a public-policy research assistant.

Use case:
- SQLite stores structured facts such as paper metadata, saved notes, and query history.
- ChromaDB stores chunks from a real PDF and answers semantic questions about the document.
- A simple router chooses the right store based on query intent.
- LangGraph checkpointers keep the conversation thread resumable.

LangChain’s SQL-agent tutorial describes the SQL agent loop of inspecting tables and schemas, deciding what is relevant, generating a query, and executing it. LangGraph’s persistence docs say checkpointers persist thread-scoped state, while stores hold durable application data across threads. LangChain’s knowledge-base tutorial explains the load-split-embed-store pattern for PDF search. 

## Why this use case works

A public-policy analyst often needs both kinds of retrieval:

- structured facts such as paper title, authors, and saved research notes
- semantic answers from the PDF itself, such as use cases, gaps, and research directions

That makes SQLite a good home for structured data and ChromaDB a good home for semantic content.

## 1) Install packages

In [ ]:
%pip install -qU python-dotenv requests pypdf     langchain langchain-core langchain-groq     langchain-chroma chromadb sentence-transformers     langgraph langgraph-checkpoint-sqlite

## 2) Load environment variables

Create a `.env` file like this:

```env
GROQ_API_KEY=your_groq_api_key
LANGSMITH_API_KEY=your_langsmith_api_key
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=database_connected_agent
```

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")
LANGSMITH_TRACING = os.getenv("LANGSMITH_TRACING", "true").lower() == "true"
LANGSMITH_PROJECT = os.getenv("LANGSMITH_PROJECT", "database_connected_agent")

print("GROQ_API_KEY set:", bool(GROQ_API_KEY))
print("LANGSMITH_API_KEY set:", bool(LANGSMITH_API_KEY))
print("LANGSMITH_TRACING:", LANGSMITH_TRACING)
print("LANGSMITH_PROJECT:", LANGSMITH_PROJECT)

## 3) Download a real PDF

We will use the public-policy paper **Explainable Machine Learning for Public Policy: Use Cases, Gaps, and Research Directions** from arXiv. The paper focuses on explainability in high-stakes policy decisions and identifies gaps in existing work. citeturn360651search0turn360651search3turn360651search12

In [ ]:
from pathlib import Path
import requests

DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

PDF_URL = "https://www.arxiv.org/pdf/2010.14374.pdf"
PDF_PATH = DATA_DIR / "public_policy_explainability.pdf"

def download_file(url: str, dest: Path) -> Path:
    if dest.exists() and dest.stat().st_size > 0:
        print("Using cached file:", dest)
        return dest
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    dest.write_bytes(resp.content)
    print("Downloaded:", dest)
    return dest

download_file(PDF_URL, PDF_PATH)

## 4) Load and chunk the PDF

LangChain’s knowledge-base tutorial describes the load → split → embed → store flow for PDF documents. 

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(str(PDF_PATH))
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(pages)

print("Pages:", len(pages))
print("Chunks:", len(chunks))
print("First page preview:")
print(pages[0].page_content[:500])

## 5) Create the ChromaDB index

Chroma is a vector store for embeddings, and LangChain provides a standard Chroma integration. 

In [ ]:
import shutil
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

CHROMA_DIR = Path("./chroma_policy_store")
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="public_policy_explainability",
    persist_directory=str(CHROMA_DIR),
)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

print("Chroma store ready at", CHROMA_DIR.resolve())

## 6) Create the SQLite database

SQLite stores the structured side of the assistant: metadata, notes, and query history.

In [ ]:
import sqlite3
from datetime import datetime

SQLITE_DB = DATA_DIR / "policy_assistant.db"
if SQLITE_DB.exists():
    SQLITE_DB.unlink()

conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

cur.execute("""
CREATE TABLE paper_metadata (
    paper_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    authors TEXT NOT NULL,
    year INTEGER NOT NULL,
    pdf_url TEXT NOT NULL,
    local_path TEXT NOT NULL,
    description TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE paper_notes (
    note_id INTEGER PRIMARY KEY AUTOINCREMENT,
    page_number INTEGER NOT NULL,
    note_text TEXT NOT NULL,
    created_at TEXT NOT NULL
)
""")

cur.execute("""
CREATE TABLE query_log (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    question TEXT NOT NULL,
    intent TEXT NOT NULL,
    source TEXT NOT NULL,
    created_at TEXT NOT NULL
)
""")

cur.execute(
    "INSERT INTO paper_metadata (title, authors, year, pdf_url, local_path, description) VALUES (?, ?, ?, ?, ?, ?)",
    (
        "Explainable Machine Learning for Public Policy: Use Cases, Gaps, and Research Directions",
        "Kasun Amarasinghe; Kit Rodolfa; Hemank Lamba; Rayid Ghani",
        2020,
        PDF_URL,
        str(PDF_PATH),
        "A public-policy research paper about explainability use cases, user goals, and research gaps.",
    ),
)

for chunk in chunks[:8]:
    cur.execute(
        "INSERT INTO paper_notes (page_number, note_text, created_at) VALUES (?, ?, ?)",
        (
            int(chunk.metadata.get("page", 0)),
            chunk.page_content[:600].replace("", " "),
            datetime.utcnow().isoformat(),
        ),
    )

conn.commit()
conn.close()

print("SQLite database created:", SQLITE_DB.resolve())

## 7) Inspect the SQLite data

In [ ]:
conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()

print("paper_metadata rows:")
for row in cur.execute("SELECT title, authors, year FROM paper_metadata"):
    print(row)

print("paper_notes sample rows:")
for row in cur.execute("SELECT page_number, substr(note_text, 1, 120) FROM paper_notes LIMIT 3"):
    print(row)

conn.close()

## 8) Set up Groq

Groq will synthesize the final answer from the retrieved context.

In [ ]:
from langchain_groq import ChatGroq

GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile")
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

print("Groq model:", GROQ_MODEL)

## 9) Define the graph state

The graph uses a small state object plus a checkpointed thread so the conversation can continue across turns. LangGraph checkpointers save thread state as checkpoints, enabling conversational memory and time travel. 

In [ ]:
import operator
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver

class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    intent: str
    context: str
    source: str
    answer: str

## 10) Routing helpers

Use SQLite for structured questions like title, authors, notes, and query history.

Use ChromaDB for semantic questions about the paper’s content, such as use cases, gaps, and research directions.

In [ ]:
def latest_user_message(state: AgentState) -> str:
    for msg in reversed(state["messages"]):
        if isinstance(msg, HumanMessage):
            return msg.content
        if isinstance(msg, dict) and msg.get("role") == "user":
            return msg.get("content", "")
    return ""

def route_intent(question: str) -> str:
    q = question.lower()
    sqlite_terms = [
        "title", "author", "year", "metadata", "note", "saved", "log", "history", "count"
    ]
    policy_terms = [
        "use case", "use cases", "gap", "gaps", "research direction", "research directions",
        "public policy", "explainable", "explanation", "paper", "document"
    ]

    if any(term in q for term in sqlite_terms):
        return "sqlite"
    if any(term in q for term in policy_terms):
        return "chroma"
    return "chroma"

## 11) Retrieve from SQLite or ChromaDB

In [ ]:
def sqlite_context(question: str) -> str:
    q = question.lower()
    conn = sqlite3.connect(SQLITE_DB)
    cur = conn.cursor()

    if any(term in q for term in ["title", "author", "year", "metadata"]):
        rows = cur.execute("SELECT title, authors, year, description FROM paper_metadata").fetchall()
        conn.close()
        return "".join([f"Title: {r[0]} | Authors: {r[1]} | Year: {r[2]} | Description: {r[3]}" for r in rows])

    if "note" in q or "saved" in q:
        rows = cur.execute("SELECT page_number, note_text FROM paper_notes ORDER BY note_id DESC LIMIT 5").fetchall()
        conn.close()
        return "".join([f"Page {r[0]}: {r[1]}" for r in rows])

    if "log" in q or "history" in q:
        rows = cur.execute("SELECT question, intent, source, created_at FROM query_log ORDER BY log_id DESC LIMIT 5").fetchall()
        conn.close()
        return "".join([f"Question: {r[0]} | intent={r[1]} | source={r[2]} | at={r[3]}" for r in rows])

    rows = cur.execute("SELECT title, authors, year FROM paper_metadata").fetchall()
    conn.close()
    return "".join([f"{r[0]} by {r[1]} ({r[2]})" for r in rows])

def chroma_context(question: str) -> str:
    docs = retriever.invoke(question)
    lines = []
    for doc in docs:
        page = doc.metadata.get("page", "n/a")
        preview = doc.page_content[:700].replace("", " ")
        lines.append(f"[page={page}] {preview}")
    return "".join(lines)

## 12) Define the answer and log nodes

In [ ]:
def route_node(state: AgentState):
    question = latest_user_message(state)
    intent = route_intent(question)
    return {"intent": intent}

def answer_node(state: AgentState):
    question = latest_user_message(state)
    context = state.get("context", "")
    source = state.get("source", "chroma")

    prompt = (
        "You are a public-policy research assistant.\n\n"
        f"Query source: {source}\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer clearly and concisely. If the context is insufficient, say so."
    )

    try:
        response = llm.invoke(prompt).content.strip()
    except Exception as e:
        response = f"LLM unavailable, fallback answer: {context[:500]} | error={e}"

    return {
        "answer": response,
        "messages": [AIMessage(content=response)],
    }

def log_node(state: AgentState):
    question = latest_user_message(state)
    conn = sqlite3.connect(SQLITE_DB)
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO query_log (question, intent, source, created_at) VALUES (?, ?, ?, ?)",
        (question, state.get("intent", ""), state.get("source", ""), datetime.utcnow().isoformat()),
    )
    conn.commit()
    conn.close()
    return {}

## 13) Build the router graph

The checkpointer stores the thread state so the same thread can be resumed later with the same `thread_id`. 

In [ ]:
def route_after_intent(state: AgentState) -> Literal["sqlite_lookup", "chroma_lookup"]:
    return "sqlite_lookup" if state["intent"] == "sqlite" else "chroma_lookup"

def sqlite_lookup_node(state: AgentState):
    question = latest_user_message(state)
    context = sqlite_context(question)
    return {"context": context, "source": "sqlite"}

def chroma_lookup_node(state: AgentState):
    question = latest_user_message(state)
    context = chroma_context(question)
    return {"context": context, "source": "chroma"}

builder = StateGraph(AgentState)

builder.add_node("route", route_node)
builder.add_node("sqlite_lookup", sqlite_lookup_node)
builder.add_node("chroma_lookup", chroma_lookup_node)
builder.add_node("answer", answer_node)
builder.add_node("log", log_node)

builder.add_edge(START, "route")
builder.add_conditional_edges("route", route_after_intent)
builder.add_edge("sqlite_lookup", "answer")
builder.add_edge("chroma_lookup", "answer")
builder.add_edge("answer", "log")
builder.add_edge("log", END)

checkpoint_db = sqlite3.connect(str(DATA_DIR / "agent_checkpoints.db"), check_same_thread=False)
checkpointer = SqliteSaver(checkpoint_db)
graph = builder.compile(checkpointer=checkpointer)

print("Graph compiled.")

## 14) Ask a structured SQLite question

In [ ]:
config = {"configurable": {"thread_id": "policy-thread-1"}}

result_sqlite = graph.invoke(
    {
        "messages": [HumanMessage(content="What is the title and who are the authors of the paper?")],
        "intent": "",
        "context": "",
        "source": "",
        "answer": "",
    },
    config=config,
)

result_sqlite

## 15) Ask a semantic PDF question

In [ ]:
result_chroma = graph.invoke(
    {
        "messages": [HumanMessage(content="What use cases does the paper identify for explainable ML in public policy?")],
        "intent": "",
        "context": "",
        "source": "",
        "answer": "",
    },
    config=config,
)

result_chroma

## 16) Ask about saved notes

In [ ]:
result_notes = graph.invoke(
    {
        "messages": [HumanMessage(content="Show me the latest saved notes from the paper.")],
        "intent": "",
        "context": "",
        "source": "",
        "answer": "",
    },
    config=config,
)

result_notes

## 17) Follow-up in the same thread

In [ ]:
follow_up = graph.invoke(
    {
        "messages": [HumanMessage(content="And what about the research gaps?")],
        "intent": "",
        "context": "",
        "source": "",
        "answer": "",
    },
    config=config,
)

follow_up

## 18) Inspect query history

In [ ]:
conn = sqlite3.connect(SQLITE_DB)
cur = conn.cursor()
rows = cur.execute("SELECT question, intent, source, created_at FROM query_log ORDER BY log_id DESC").fetchall()
conn.close()

rows

## Key takeaways

- SQLite is the structured store for metadata, notes, and query history.
- ChromaDB is the semantic store for the PDF content.
- A simple router can send a query to the right store based on intent.
- LangGraph checkpointers keep the thread resumable, so the assistant can continue the same conversation later. 

## References

- SQL agent: https://docs.langchain.com/oss/python/langchain/sql-agent
- Persistence and checkpointers: https://docs.langchain.com/oss/python/langgraph/persistence
- Chroma integration: https://docs.langchain.com/oss/python/integrations/vectorstores/chroma
- Knowledge base tutorial: https://docs.langchain.com/oss/python/langchain/knowledge-base